# Stack 1 : RAG traditionnel avec FAISS

Ce notebook implémente un pipeline RAG traditionnel complet :

- **Données** : Simple English Wikipedia (100 articles)
- **Chunking** : découpage récursif avec chevauchement (`src/shared/chunker.py`)
- **Embeddings** : `all-MiniLM-L6-v2` (384 dimensions)
- **Index** : FAISS avec similarité cosinus (produit scalaire après normalisation L2)
- **Génération** : Ollama (`llama3.1` par défaut)
- **Évaluation** : métriques RAGAS

C'est le stack de référence à recherche vectorielle auquel sont comparés les stacks suivants de la série.

## 1. Configuration

In [ ]:
import sys
import time
import json
from pathlib import Path

import numpy as np

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from shared.chunker import recursive_chunk, Chunk
from shared.embeddings import EmbeddingModel
from shared.llm import call_llm

print(f"Racine du projet : {PROJECT_ROOT}")

## 2. Chargement des données

In [ ]:
from datasets import load_dataset

ds = load_dataset("wikimedia/wikipedia", "20231101.simple", split="train[:100]")
print(f"{len(ds)} articles chargés")
print(f"Premier article : '{ds[0]['title']}' ({len(ds[0]['text'])} caractères)")

## 3. Chunking

Nous utilisons le découpage récursif avec un chevauchement de 50 caractères pour
préserver le contexte aux frontières de chunks. Voir `01_chunking.ipynb` pour une
comparaison détaillée des stratégies de chunking.

In [ ]:
all_chunks: list[Chunk] = []

for doc in ds:
    chunks = recursive_chunk(
        doc['text'],
        max_size=500,
        overlap=50,
        metadata={'title': doc['title'], 'url': doc['url']},
    )
    all_chunks.extend(chunks)

print(f"Nombre total de chunks : {len(all_chunks)}")
print(f"Taille moyenne des chunks : {np.mean([len(c.text) for c in all_chunks]):.0f} caractères")
print(f"\nChunk exemple :")
print(f"  Texte : {all_chunks[0].text[:120]}...")
print(f"  Métadonnées : {all_chunks[0].metadata}")

## 4. Embeddings

Encode tous les chunks en vecteurs denses à l'aide de `all-MiniLM-L6-v2`.

In [ ]:
emb_model = EmbeddingModel("all-MiniLM-L6-v2")
print(f"Dimension des embeddings : {emb_model.dimension}")

chunk_texts = [c.text for c in all_chunks]
embeddings = emb_model.encode(chunk_texts)
print(f"Forme des embeddings : {embeddings.shape}")

## 5. Indexation FAISS

Construit un index FAISS utilisant la similarité par produit scalaire (cosinus
après normalisation L2). C'est une recherche exacte, adaptée à des jeux de
données allant jusqu'à quelques centaines de milliers de chunks.

In [ ]:
from shared.vector_index import FaissIndexer

indexer = FaissIndexer(dimension=emb_model.dimension)

chunk_metadata = [c.metadata for c in all_chunks]
indexer.add(embeddings, chunk_texts, chunk_metadata)

print(f"Taille de l'index FAISS : {indexer.size} vecteurs")

# Sauvegarde l'index pour réutilisation
index_path = PROJECT_ROOT / "data" / "corpus" / "faiss_index"
indexer.save(str(index_path))
print(f"Index sauvegardé dans : {index_path}")

## 6. Fonction de recherche

Le récupérateur encode la requête, recherche dans l'index FAISS et renvoie les
K chunks les plus similaires.

In [ ]:
from stack1_traditional import VectorRetriever

retriever = VectorRetriever(indexer, emb_model)

# Test de recherche
for i, r in enumerate(retriever.search("Which science studies plants?", k=5)):
    print(f"Résultat {i+1} (score={r['score']:.4f}) : [{r['metadata'].get('title', '')}] {r['text'][:80]}...")

## 7. Pipeline RAG

Combine récupération et génération : recherche les chunks pertinents, construit
un prompt avec le contexte, puis génère une réponse à l'aide du LLM.

In [ ]:
from stack1_traditional import TraditionalRAG

rag = TraditionalRAG(retriever, llm_fn=call_llm)

# Le gabarit de prompt est défini une seule fois dans src/shared/prompts.py
# et partagé par les trois stacks (variable tenue constante)
print(rag.prompt_template)

In [ ]:
# Test du pipeline RAG complet
question = "What happened in April 1912?"
result = rag.query(question, k=5)

print(f"Question : {question}")
print(f"Réponse : {result['answer']}")
print(f"\nRécupération : {result['retrieval_ms']} ms")
print(f"Génération  : {result['generation_ms']} ms")
print(f"Total       : {result['latency_ms']} ms")
sources = [c['metadata'].get('title', '') for c in result['contexts']]
print(f"Sources : {sources}")

In [ ]:
# Test sur plusieurs requêtes
test_queries = [
    "What happened in April 1912?",
    "Which science studies plants?",
    "What is the capital of France?",
    "Who invented the telephone?",
    "What is the largest ocean?",
]

for q in test_queries:
    r = rag.query(q)
    print(f"Q : {q}")
    print(f"R : {r['answer']}")
    print(f"   [{r['latency_ms']} ms]")
    print()

## 8. Évaluation RAGAS

Évalue la qualité de notre pipeline RAG à l'aide des métriques RAGAS :
- **Faithfulness** : la réponse est-elle factuellement cohérente avec le contexte récupéré ?
- **Answer Relevancy** : dans quelle mesure la réponse est-elle pertinente par rapport à la question ?
- **Context Precision** : quelle fraction des contextes récupérés est pertinente ?
- **Context Recall** : quelle fraction de l'information pertinente a été récupérée ?

**Remarque** : RAGAS nécessite des réponses de référence (ground truth). Si
`eval/questions.json` existe, nous les utilisons ; sinon nous fournissons des
réponses de référence d'exemple.

In [ ]:
# Charge ou définit les questions d'évaluation
eval_path = PROJECT_ROOT / "eval" / "questions.json"

if eval_path.exists():
    with open(eval_path, encoding='utf-8') as f:
        eval_data = json.load(f)
    print(f"{len(eval_data)} questions d'évaluation chargées depuis {eval_path}")
else:
    # Fournit des données d'évaluation d'exemple
    eval_data = [
        {
            "question": "What happened in April 1912?",
            "ground_truth": "The RMS Titanic sank near Newfoundland after hitting an iceberg on April 15, 1912."
        },
        {
            "question": "Which science studies plants?",
            "ground_truth": "Botany is the science that studies plants. It is a branch of biology."
        },
        {
            "question": "What is the capital of France?",
            "ground_truth": "Paris is the capital of France."
        },
    ]
    # Sauvegarde pour réutilisation par les autres notebooks
    eval_path.parent.mkdir(parents=True, exist_ok=True)
    with open(eval_path, 'w', encoding='utf-8') as f:
        json.dump(eval_data, f, indent=2, ensure_ascii=False)
    print(f"Fichier d'évaluation d'exemple créé dans {eval_path}")

In [ ]:
# Exécuter le RAG sur les questions d'évaluation
eval_questions = [e['question'] for e in eval_data]
eval_ground_truths = [e['ground_truth'] for e in eval_data]

eval_answers = []
eval_contexts = []

for q in eval_questions:
    r = rag.query(q)
    eval_answers.append(r['answer'])
    # RAGAS attend les contextes sous forme de liste de textes
    eval_contexts.append([c['text'] for c in r['contexts']])

print(f"Réponses générées pour {len(eval_questions)} questions.")
for i, (q, a) in enumerate(zip(eval_questions, eval_answers)):
    print(f"  Q{i+1} : {q}")
    print(f"  R{i+1} : {a}")
    print()

In [ ]:
# Lance l'évaluation RAGAS
try:
    from shared.evaluator import evaluate_rag

    ragas_results = evaluate_rag(
        questions=eval_questions,
        answers=eval_answers,
        contexts=eval_contexts,
        ground_truths=eval_ground_truths,
    )

    print("=" * 50)
    print("Résultats de l'évaluation RAGAS (Stack 1 : RAG traditionnel)")
    print("=" * 50)
    for metric in ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']:
        print(f"  {metric} : {ragas_results.get(metric, 'N/A'):.4f}")

    # Sauvegarde les résultats
    eval_output_path = PROJECT_ROOT / "eval" / "traditional_eval.json"
    with open(eval_output_path, 'w', encoding='utf-8') as f:
        json.dump(ragas_results, f, indent=2, default=str)
    print(f"\nRésultats sauvegardés dans {eval_output_path}")

except Exception as exc:
    print(f"L'évaluation RAGAS a échoué : {exc}")
    print("Cela peut nécessiter une configuration supplémentaire (par ex. une clé API OpenAI pour RAGAS).")
    print("Vous pouvez tout de même utiliser le pipeline RAG ci-dessus sans RAGAS.")

## Résumé

Le Stack 1 (RAG traditionnel) fournit une base solide :
- Récupération rapide via la recherche exacte FAISS
- Bonne qualité grâce au découpage récursif avec chevauchement
- Architecture simple, sans services externes au-delà d'Ollama

**Limitations** :
- La recherche purement vectorielle peut manquer les correspondances exactes de mots-clés
- Aucune compréhension des relations entre entités à travers les documents

Les stacks hybride et graphe, construits plus tard dans la série, répondent à ces limitations.